<a href="https://colab.research.google.com/github/mbachant/Character-Complexity/blob/main/Plotting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CSV Files Guide

### `text_complexity_by_font.csv`
Contains the **overall complexity score for each text in each usable font**.

Use this file to compare the visual complexity of different languages or writing systems.

- One row = one text × one font
- `normalized_complexity` = main complexity score to use for comparisons
- `total_complexity` = total complexity before correcting for text length

### `character_complexity_by_text_font.csv`
Contains the **individual character data used to calculate the text complexity scores**.

Use this file to examine which characters contribute to a text's complexity.

- One row = one character × one text × one font
- `count` = how often the character occurs
- `complexity` = visual complexity of that character
- `weighted_complexity` = `count × complexity`

### `english_representation_comparison.csv`
Contains the **English font, Morse Code, and Braille text-level complexity scores**.

Use this file to compare ordinary written English with Morse and Braille.

Morse and Braille are compared separately with English. If a character is not supported by Morse or Braille, it is removed from both that representation and its matching English calculation so the comparison uses the same source characters.

### `english_representation_character_complexity.csv`
Contains the **character-level data behind the English, Morse, and Braille comparison**.

Use this file to see how individual English characters and their Morse or Braille representations contribute to the overall complexity scores.

- One row = one source character in one representation
- Includes character frequency, complexity, and weighted complexity
- Morse sequences and multi-cell Braille representations are treated as representations of a single source character

In [11]:
# ============================================================
# INSTALL NOTO FONTS FOR PLOTTING
# ============================================================

!apt-get update -qq
!apt-get install -y fonts-noto fonts-noto-cjk > /dev/null

print("Noto fonts installed.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Noto fonts installed.


In [2]:
# ============================================================
# INPUT / OUTPUT SETTINGS
# ============================================================

# Choose input/output locations
#
# "drive"    = Google Drive
# "computer" = upload/download using your computer

input_source = "drive"
output_source = "drive"


base_dir = (
    "/content/drive/MyDrive/Character Complexity"
)


# ------------------------------------------------------------
# Input CSV filenames
# ------------------------------------------------------------

text_complexity_filename = (
    "text_complexity_by_font.csv"
)

character_complexity_filename = (
    "character_complexity_by_text_font.csv"
)

english_representation_comparison_filename = (
    "english_representation_comparison.csv"
)

english_representation_character_filename = (
    "english_representation_character_complexity.csv"
)

In [3]:
# ============================================================
# PATHS / LOAD INPUT FILES
# ============================================================

import os
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive, files


# ------------------------------------------------------------
# Mount Google Drive if needed
# ------------------------------------------------------------

if (
    input_source == "drive"
    or output_source == "drive"
):

    drive.mount(
        "/content/drive",
        force_remount=False
    )


input_dir = (
    base_dir
    if input_source == "drive"
    else "/content"
)


output_dir = (
    base_dir
    if output_source == "drive"
    else "/content"
)


# ------------------------------------------------------------
# Input paths
# ------------------------------------------------------------

text_complexity_path = os.path.join(
    input_dir,
    text_complexity_filename
)


character_complexity_path = os.path.join(
    input_dir,
    character_complexity_filename
)


english_representation_comparison_path = os.path.join(
    input_dir,
    english_representation_comparison_filename
)


english_representation_character_path = os.path.join(
    input_dir,
    english_representation_character_filename
)


# ------------------------------------------------------------
# Upload from computer if requested
# ------------------------------------------------------------

if input_source == "computer":

    print(
        "Upload the four text-complexity CSV files."
    )

    uploaded = files.upload()


    required_filenames = [
        text_complexity_filename,
        character_complexity_filename,
        english_representation_comparison_filename,
        english_representation_character_filename
    ]


    missing_files = [
        filename
        for filename in required_filenames
        if filename not in uploaded
    ]


    if missing_files:

        raise FileNotFoundError(
            "Missing required CSV files:\n"
            + "\n".join(
                f"  {filename}"
                for filename in missing_files
            )
        )


    for filename in required_filenames:

        file_path = os.path.join(
            input_dir,
            filename
        )

        with open(
            file_path,
            "wb"
        ) as f:

            f.write(
                uploaded[
                    filename
                ]
            )

Mounted at /content/drive


In [4]:
# ============================================================
# LOAD + VERIFY TEXT COMPLEXITY DATASETS
# ============================================================


# ------------------------------------------------------------
# Verify files exist
# ------------------------------------------------------------

required_paths = {
    "Text × font complexity":
        text_complexity_path,

    "Character × text × font complexity":
        character_complexity_path,

    "English representation comparison":
        english_representation_comparison_path,

    "English representation character complexity":
        english_representation_character_path
}


for dataset_name, path in required_paths.items():

    if not os.path.isfile(
        path
    ):

        raise FileNotFoundError(
            f"{dataset_name} file not found:\n"
            f"{path}"
        )


# ------------------------------------------------------------
# Load CSVs
# ------------------------------------------------------------

text_complexity = pd.read_csv(
    text_complexity_path
)


character_complexity = pd.read_csv(
    character_complexity_path
)


english_representation_comparison = pd.read_csv(
    english_representation_comparison_path
)


english_representation_character_complexity = (
    pd.read_csv(
        english_representation_character_path
    )
)


# ------------------------------------------------------------
# Verify required columns
# ------------------------------------------------------------

required_text_columns = {
    "language",
    "language_name",
    "font_id",
    "font_name",
    "total_complexity",
    "normalized_complexity"
}


required_character_columns = {
    "language",
    "language_name",
    "font_id",
    "font_name",
    "character",
    "codepoint",
    "count",
    "complexity",
    "weighted_complexity"
}


required_representation_columns = {
    "language",
    "language_name",
    "comparison_system",
    "representation_type",
    "representation_name",
    "normalized_complexity",
    "total_complexity",
    "matched_characters",
    "matched_unique_characters"
}


required_representation_character_columns = {
    "language",
    "language_name",
    "comparison_system",
    "representation_type",
    "representation_name",
    "character",
    "codepoint",
    "count",
    "complexity",
    "weighted_complexity"
}


dataset_checks = [
    (
        "Text × font complexity",
        text_complexity,
        required_text_columns
    ),
    (
        "Character × text × font complexity",
        character_complexity,
        required_character_columns
    ),
    (
        "English representation comparison",
        english_representation_comparison,
        required_representation_columns
    ),
    (
        "English representation character complexity",
        english_representation_character_complexity,
        required_representation_character_columns
    )
]


for (
    dataset_name,
    dataframe,
    required_columns
) in dataset_checks:

    missing_columns = (
        required_columns
        - set(
            dataframe.columns
        )
    )

    if missing_columns:

        raise ValueError(
            f"{dataset_name} is missing "
            f"required columns: "
            f"{sorted(missing_columns)}"
        )


# ------------------------------------------------------------
# Report loaded datasets
# ------------------------------------------------------------

print()
print("=" * 70)
print("TEXT COMPLEXITY DATASETS")
print("=" * 70)

print()
print("TEXT × FONT COMPLEXITY")

print(
    f"  Path: "
    f"{text_complexity_path}"
)

print(
    f"  Rows: "
    f"{len(text_complexity):,}"
)

print(
    f"  Languages: "
    f"{text_complexity['language'].nunique():,}"
)

print(
    f"  Fonts: "
    f"{text_complexity['font_id'].nunique():,}"
)


print()
print("CHARACTER × TEXT × FONT COMPLEXITY")

print(
    f"  Path: "
    f"{character_complexity_path}"
)

print(
    f"  Rows: "
    f"{len(character_complexity):,}"
)


print()
print("ENGLISH REPRESENTATION COMPARISON")

print(
    f"  Path: "
    f"{english_representation_comparison_path}"
)

print(
    f"  Rows: "
    f"{len(english_representation_comparison):,}"
)

print(
    f"  Comparison systems: "
    f"{english_representation_comparison['comparison_system'].nunique():,}"
)


print()
print("ENGLISH REPRESENTATION CHARACTER COMPLEXITY")

print(
    f"  Path: "
    f"{english_representation_character_path}"
)

print(
    f"  Rows: "
    f"{len(english_representation_character_complexity):,}"
)


TEXT COMPLEXITY DATASETS

TEXT × FONT COMPLEXITY
  Path: /content/drive/MyDrive/Character Complexity/text_complexity_by_font.csv
  Rows: 1,123
  Languages: 236
  Fonts: 5

CHARACTER × TEXT × FONT COMPLEXITY
  Path: /content/drive/MyDrive/Character Complexity/character_complexity_by_text_font.csv
  Rows: 67,895

ENGLISH REPRESENTATION COMPARISON
  Path: /content/drive/MyDrive/Character Complexity/english_representation_comparison.csv
  Rows: 12
  Comparison systems: 2

ENGLISH REPRESENTATION CHARACTER COMPLEXITY
  Path: /content/drive/MyDrive/Character Complexity/english_representation_character_complexity.csv
  Rows: 666


In [12]:
# ============================================================
# CONFIGURE MATPLOTLIB FOR UNICODE CHARACTERS
# ============================================================

# Use Noto Sans fonts for plotting.
#
# This helps prevent missing-glyph rectangles when plotting
# characters from CJK and other Unicode writing systems.

import matplotlib
import matplotlib.pyplot as plt

from matplotlib import font_manager


# ------------------------------------------------------------
# Find Noto Sans fonts
# ------------------------------------------------------------

noto_font_paths = [
    path
    for path in font_manager.findSystemFonts()
    if "NotoSans" in path.replace("-", "")
]


print(
    f"Noto Sans font files found: "
    f"{len(noto_font_paths):,}"
)


# ------------------------------------------------------------
# Register discovered Noto fonts
# ------------------------------------------------------------

for font_path in noto_font_paths:

    try:

        font_manager.fontManager.addfont(
            font_path
        )

    except Exception:

        pass


# ------------------------------------------------------------
# Select Unicode-capable default font
# ------------------------------------------------------------

preferred_fonts = [
    "Noto Sans CJK JP",
    "Noto Sans",
    "Noto Sans JP",
    "Noto Sans CJK"
]


available_font_names = {
    font.name
    for font in font_manager.fontManager.ttflist
}


selected_default_font = None


for font_name in preferred_fonts:

    if font_name in available_font_names:

        selected_default_font = font_name
        break


if selected_default_font is None:

    raise RuntimeError(
        "Could not find a suitable Noto Sans font."
    )


# ------------------------------------------------------------
# Set Matplotlib defaults
# ------------------------------------------------------------

matplotlib.rcParams[
    "font.family"
] = [
    selected_default_font
]


matplotlib.rcParams[
    "axes.unicode_minus"
] = False


print(
    f"Matplotlib default font: "
    f"{selected_default_font}"
)

Noto Sans font files found: 1,530
Matplotlib default font: Noto Sans CJK JP


In [14]:
# ============================================================
# CREATE PLOTTING LOOKUPS FROM CSV DATA
# ============================================================


# ------------------------------------------------------------
# Language table
#
# language_index is the handle that can be used in later
# plotting / analysis cells.
# ------------------------------------------------------------

language_table = (
    text_complexity[
        [
            "language",
            "language_name"
        ]
    ]
    .drop_duplicates(
        subset=["language"]
    )
    .sort_values(
        [
            "language_name",
            "language"
        ]
    )
    .reset_index(
        drop=True
    )
)


language_table.insert(
    0,
    "language_index",
    range(
        len(language_table)
    )
)


# ------------------------------------------------------------
# Add number of usable fonts per language
# ------------------------------------------------------------

usable_fonts_per_language = (
    text_complexity
    .groupby(
        "language"
    )["font_id"]
    .nunique()
    .rename(
        "usable_fonts"
    )
    .reset_index()
)


language_table = (
    language_table
    .merge(
        usable_fonts_per_language,
        on="language",
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# Font table
#
# font_index can also be used as a handle in later cells.
# ------------------------------------------------------------

font_table = (
    text_complexity[
        [
            "font_id",
            "font_name"
        ]
    ]
    .drop_duplicates(
        subset=["font_id"]
    )
    .sort_values(
        "font_name"
    )
    .reset_index(
        drop=True
    )
)


font_table.insert(
    0,
    "font_index",
    range(
        len(font_table)
    )
)


# ------------------------------------------------------------
# Analysis font list
# ------------------------------------------------------------

analysis_fonts = (
    font_table[
        "font_id"
    ]
    .tolist()
)


# ------------------------------------------------------------
# Name lookups
# ------------------------------------------------------------

language_names = (
    language_table
    .set_index(
        "language"
    )["language_name"]
)


font_names = (
    font_table
    .set_index(
        "font_id"
    )["font_name"]
)


# ------------------------------------------------------------
# Index lookups
# ------------------------------------------------------------

language_by_index = (
    language_table
    .set_index(
        "language_index"
    )
)


font_by_index = (
    font_table
    .set_index(
        "font_index"
    )
)


# ------------------------------------------------------------
# Search functions
# ------------------------------------------------------------

def search_languages(
    search_term
):

    search_term = str(
        search_term
    ).strip()


    results = (
        language_table[
            language_table[
                "language_name"
            ]
            .str.contains(
                search_term,
                case=False,
                na=False,
                regex=False
            )
            |
            language_table[
                "language"
            ]
            .str.contains(
                search_term,
                case=False,
                na=False,
                regex=False
            )
        ]
        .copy()
    )


    return results


def search_fonts(
    search_term
):

    search_term = str(
        search_term
    ).strip()


    results = (
        font_table[
            font_table[
                "font_name"
            ]
            .str.contains(
                search_term,
                case=False,
                na=False,
                regex=False
            )
        ]
        .copy()
    )


    return results


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("=" * 70)
print("PLOTTING LOOKUPS")
print("=" * 70)

print()

print(
    f"Languages available for plotting: "
    f"{len(language_table):,}"
)

print(
    f"Fonts available for plotting: "
    f"{len(font_table):,}"
)


print()
print("FONTS")
print("-" * 70)

display(
    font_table
)


PLOTTING LOOKUPS

Languages available for plotting: 236
Fonts available for plotting: 5

FONTS
----------------------------------------------------------------------


,font_index,font_id,font_name
0,0,2142,Noto Sans CJK SC
1,1,2288,Noto Sans Mono Condensed Black
2,2,1344,Noto Sans Mono Condensed Light
3,3,2213,Noto Sans Mono ExtraCondensed SemiBold
4,4,2211,Noto Sans Mono ExtraCondensed Thin


In [15]:
# ============================================================
# BROWSE / SEARCH LANGUAGES
# ============================================================

# Show all available languages and their index numbers.

display(
    language_table
)

,language_index,language,language_name,usable_fonts
0,0,abk,Abkhaz,5
1,1,atj,Achehnese,5
2,2,acu,Achuar-Shiwiar,5
3,3,gax,Afaan Oromo (Oromiffa),5
4,4,afk,Afrikaans,5
...,...,...,...,...
231,231,frn1,Walloon/Wallon,5
232,232,guc,Wayuu,5
233,233,tsw,Western Sotho/Tswana/Setswana,4
234,234,ccx,Zhuang,5


In [18]:
# ============================================================
# SEARCH LANGUAGES
# ============================================================

# Search by language name or language code.


search_term = "rus"


display(
    search_languages(
        search_term
    )
)

,language_index,language,language_name,usable_fonts
23,23,ruw,Belorus (Belaruski),4
187,187,rus,Russian (Russky),5


In [19]:
# ============================================================
# INTERACTIVE RAW TEXT COMPLEXITY BY LANGUAGE
# ============================================================

import plotly.express as px


# ------------------------------------------------------------
# Add language index to text-complexity data
# ------------------------------------------------------------

plot_data = (
    text_complexity
    .merge(
        language_table[
            [
                "language_index",
                "language"
            ]
        ],
        on="language",
        how="left",
        validate="many_to_one"
    )
    .dropna(
        subset=[
            "language_index",
            "total_complexity",
            "font_name"
        ]
    )
    .copy()
)


plot_data[
    "language_index"
] = (
    plot_data[
        "language_index"
    ]
    .astype(int)
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig = px.scatter(
    plot_data,
    x="language_index",
    y="total_complexity",
    color="font_name",
    hover_name="language_name",
    hover_data={
        "language_index": True,
        "language": True,
        "font_name": True,
        "total_complexity": ":.6f"
    },
    labels={
        "language_index":
            "Language index",
        "total_complexity":
            "Raw total complexity",
        "font_name":
            "Font"
    },
    title=(
        "Raw Text Complexity by Language"
    )
)


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

fig.update_traces(
    marker=dict(
        size=8,
        opacity=0.75
    )
)


fig.update_layout(
    width=1200,
    height=700,

    legend=dict(
        title="Font",
        x=1.02,
        y=1
    ),

    margin=dict(
        r=220
    )
)


fig.update_xaxes(
    title="Language index",
    dtick=10,
    showgrid=True
)


fig.update_yaxes(
    title="Raw total complexity",
    showgrid=True
)


fig.show(
    config={
        "scrollZoom": True,
        "displaylogo": False
    }
)

In [23]:
# ============================================================
# INTERACTIVE NORMALIZED TEXT COMPLEXITY BY LANGUAGE
# ============================================================


plot_data = (
    text_complexity
    .merge(
        language_table[
            [
                "language_index",
                "language"
            ]
        ],
        on="language",
        how="left",
        validate="many_to_one"
    )
    .dropna(
        subset=[
            "language_index",
            "normalized_complexity",
            "font_name"
        ]
    )
    .copy()
)


plot_data[
    "language_index"
] = (
    plot_data[
        "language_index"
    ]
    .astype(int)
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig = px.scatter(
    plot_data,
    x="language_index",
    y="normalized_complexity",
    color="font_name",
    hover_name="language_name",
    hover_data={
        "language_index": True,
        "language": True,
        "font_name": True,
        "normalized_complexity": ":.6f"
    },
    labels={
        "language_index":
            "Language index",
        "normalized_complexity":
            "Normalized complexity",
        "font_name":
            "Font"
    },
    title=(
        "Normalized Text Complexity by Language"
    )
)


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

fig.update_traces(
    marker=dict(
        size=8,
        opacity=0.75
    )
)


fig.update_layout(
    width=1200,
    height=700,

    legend=dict(
        title="Font",
        x=1.02,
        y=1
    ),

    margin=dict(
        r=220
    )
)


fig.update_xaxes(
    title="Language index",
    dtick=10,
    showgrid=True
)


fig.update_yaxes(
    title="Normalized complexity",
    showgrid=True
)


fig.show(
    config={
        "scrollZoom": True,
        "displaylogo": False
    }
)

In [21]:
# ============================================================
# INTERACTIVE RAW COMPLEXITY VS TEXT LENGTH
# ============================================================

# Each point represents one language × font observation.
#
# x = number of non-whitespace source-character occurrences
# y = total raw complexity
# color = font
#
# Language index is shown in the hover information and can be
# used as a handle for selecting languages in later analyses.


# ------------------------------------------------------------
# Prepare plot data
# ------------------------------------------------------------

plot_data = (
    text_complexity
    .merge(
        language_table[
            [
                "language_index",
                "language"
            ]
        ],
        on="language",
        how="left",
        validate="many_to_one"
    )
    .dropna(
        subset=[
            "language_index",
            "nonwhitespace_characters",
            "total_complexity",
            "font_name"
        ]
    )
    .copy()
)


plot_data[
    "language_index"
] = (
    plot_data[
        "language_index"
    ]
    .astype(int)
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig = px.scatter(
    plot_data,
    x="nonwhitespace_characters",
    y="total_complexity",
    color="font_name",
    hover_name="language_name",
    hover_data={
        "language_index": True,
        "language": True,
        "font_name": True,
        "nonwhitespace_characters": ":,",
        "total_complexity": ":.6f"
    },
    labels={
        "nonwhitespace_characters":
            "Text length (non-whitespace characters)",
        "total_complexity":
            "Raw total complexity",
        "font_name":
            "Font",
        "language_index":
            "Language index",
        "language":
            "Language code"
    },
    title=(
        "Raw Text Complexity vs. Text Length"
    )
)


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

fig.update_traces(
    marker=dict(
        size=8,
        opacity=0.70
    )
)


fig.update_layout(
    width=1200,
    height=700,

    legend=dict(
        title="Font",
        x=1.02,
        y=1
    ),

    margin=dict(
        r=220
    )
)


fig.update_xaxes(
    title=(
        "Text length "
        "(non-whitespace characters)"
    ),
    showgrid=True
)


fig.update_yaxes(
    title="Raw total complexity",
    showgrid=True
)


fig.show(
    config={
        "scrollZoom": True,
        "displaylogo": False
    }
)

In [22]:
# ============================================================
# INTERACTIVE NORMALIZED COMPLEXITY VS TEXT LENGTH
# ============================================================

# Each point represents one language × font observation.
#
# x = number of non-whitespace source-character occurrences
# y = normalized complexity
# color = font
#
# Because normalized complexity is divided by the number of
# non-whitespace character occurrences, this plot can be used
# to examine whether the normalized metric still varies
# systematically with text length.


# ------------------------------------------------------------
# Prepare plot data
# ------------------------------------------------------------

plot_data = (
    text_complexity
    .merge(
        language_table[
            [
                "language_index",
                "language"
            ]
        ],
        on="language",
        how="left",
        validate="many_to_one"
    )
    .dropna(
        subset=[
            "language_index",
            "nonwhitespace_characters",
            "normalized_complexity",
            "font_name"
        ]
    )
    .copy()
)


plot_data[
    "language_index"
] = (
    plot_data[
        "language_index"
    ]
    .astype(int)
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig = px.scatter(
    plot_data,
    x="nonwhitespace_characters",
    y="normalized_complexity",
    color="font_name",
    hover_name="language_name",
    hover_data={
        "language_index": True,
        "language": True,
        "font_name": True,
        "nonwhitespace_characters": ":,",
        "normalized_complexity": ":.6f"
    },
    labels={
        "nonwhitespace_characters":
            "Text length (non-whitespace characters)",
        "normalized_complexity":
            "Normalized complexity",
        "font_name":
            "Font",
        "language_index":
            "Language index",
        "language":
            "Language code"
    },
    title=(
        "Normalized Text Complexity vs. Text Length"
    )
)


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

fig.update_traces(
    marker=dict(
        size=8,
        opacity=0.70
    )
)


fig.update_layout(
    width=1200,
    height=700,

    legend=dict(
        title="Font",
        x=1.02,
        y=1
    ),

    margin=dict(
        r=220
    )
)


fig.update_xaxes(
    title=(
        "Text length "
        "(non-whitespace characters)"
    ),
    showgrid=True
)


fig.update_yaxes(
    title="Normalized complexity",
    showgrid=True
)


fig.show(
    config={
        "scrollZoom": True,
        "displaylogo": False
    }
)

In [25]:
# ============================================================
# INTERACTIVE RAW TEXT COMPLEXITY BY LANGUAGE
# ============================================================

# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------

# Choose one font using its font_id.
#
# To see available fonts:
# display(font_table)

selected_font_id = analysis_fonts[0]


# ------------------------------------------------------------
# Prepare plot data
# ------------------------------------------------------------

plot_data = (
    text_complexity[
        text_complexity[
            "font_id"
        ] == selected_font_id
    ]
    .merge(
        language_table[
            [
                "language_index",
                "language"
            ]
        ],
        on="language",
        how="left",
        validate="many_to_one"
    )
    .dropna(
        subset=[
            "language_index",
            "total_complexity"
        ]
    )
    .copy()
)


plot_data[
    "language_index"
] = (
    plot_data[
        "language_index"
    ]
    .astype(int)
)


# ------------------------------------------------------------
# Sort languages from least → most complex
# ------------------------------------------------------------

plot_data = (
    plot_data
    .sort_values(
        [
            "total_complexity",
            "language_index"
        ],
        ascending=True
    )
    .reset_index(
        drop=True
    )
)


plot_data[
    "complexity_rank"
] = (
    range(
        1,
        len(plot_data) + 1
    )
)


# ------------------------------------------------------------
# Get font name
# ------------------------------------------------------------

selected_font_name = (
    font_names.loc[
        selected_font_id
    ]
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig = px.scatter(
    plot_data,
    x="complexity_rank",
    y="total_complexity",
    text="language_index",
    hover_name="language_name",
    hover_data={
        "language_index": True,
        "language": True,
        "complexity_rank": True,
        "total_complexity": ":.6f"
    },
    labels={
        "complexity_rank":
            "Languages ranked from least to most complex",
        "total_complexity":
            "Raw total complexity",
        "language_index":
            "Language index",
        "language":
            "Language code"
    },
    title=(
        "Raw Text Complexity by Language"
        f"<br>{selected_font_name}"
    )
)


# ------------------------------------------------------------
# Number labels
# ------------------------------------------------------------

fig.update_traces(
    marker=dict(
        size=7,
        opacity=0.75
    ),
    textposition="top center",
    textfont=dict(
        size=9
    )
)


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

fig.update_layout(
    width=1400,
    height=750,
    showlegend=False
)


fig.update_xaxes(
    title=(
        "Languages ranked from least "
        "to most complex"
    ),
    showticklabels=False,
    showgrid=True
)


fig.update_yaxes(
    title="Raw total complexity",
    showgrid=True
)


fig.show(
    config={
        "scrollZoom": True,
        "displaylogo": False
    }
)


# ------------------------------------------------------------
# Language legend
# ------------------------------------------------------------

print()
print("LANGUAGE INDEX")
print("-" * 70)

display(
    plot_data[
        [
            "language_index",
            "language",
            "language_name"
        ]
    ]
    .sort_values(
        "language_index"
    )
)


LANGUAGE INDEX
----------------------------------------------------------------------


,language_index,language,language_name
121,0,abk,Abkhaz
142,1,atj,Achehnese
168,2,acu,Achuar-Shiwiar
69,3,gax,Afaan Oromo (Oromiffa)
53,4,afk,Afrikaans
...,...,...,...
63,230,rmy1,Vlach
107,231,frn1,Walloon/Wallon
77,232,guc,Wayuu
49,234,ccx,Zhuang


In [26]:
# ============================================================
# INTERACTIVE NORMALIZED TEXT COMPLEXITY BY LANGUAGE
# ============================================================

# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------

# Choose one font using its font_id.
#
# To see available fonts:
# display(font_table)

selected_font_id = analysis_fonts[0]


# ------------------------------------------------------------
# Prepare plot data
# ------------------------------------------------------------

plot_data = (
    text_complexity[
        text_complexity[
            "font_id"
        ] == selected_font_id
    ]
    .merge(
        language_table[
            [
                "language_index",
                "language"
            ]
        ],
        on="language",
        how="left",
        validate="many_to_one"
    )
    .dropna(
        subset=[
            "language_index",
            "normalized_complexity"
        ]
    )
    .copy()
)


plot_data[
    "language_index"
] = (
    plot_data[
        "language_index"
    ]
    .astype(int)
)


# ------------------------------------------------------------
# Sort languages from least → most complex
# ------------------------------------------------------------

plot_data = (
    plot_data
    .sort_values(
        [
            "normalized_complexity",
            "language_index"
        ],
        ascending=True
    )
    .reset_index(
        drop=True
    )
)


plot_data[
    "complexity_rank"
] = (
    range(
        1,
        len(plot_data) + 1
    )
)


# ------------------------------------------------------------
# Get font name
# ------------------------------------------------------------

selected_font_name = (
    font_names.loc[
        selected_font_id
    ]
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig = px.scatter(
    plot_data,
    x="complexity_rank",
    y="normalized_complexity",
    text="language_index",
    hover_name="language_name",
    hover_data={
        "language_index": True,
        "language": True,
        "complexity_rank": True,
        "normalized_complexity": ":.6f"
    },
    labels={
        "complexity_rank":
            "Languages ranked from least to most complex",
        "normalized_complexity":
            "Normalized complexity",
        "language_index":
            "Language index",
        "language":
            "Language code"
    },
    title=(
        "Normalized Text Complexity by Language"
        f"<br>{selected_font_name}"
    )
)


# ------------------------------------------------------------
# Number labels
# ------------------------------------------------------------

fig.update_traces(
    marker=dict(
        size=7,
        opacity=0.75
    ),
    textposition="top center",
    textfont=dict(
        size=9
    )
)


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

fig.update_layout(
    width=1400,
    height=750,
    showlegend=False
)


fig.update_xaxes(
    title=(
        "Languages ranked from least "
        "to most complex"
    ),
    showticklabels=False,
    showgrid=True
)


fig.update_yaxes(
    title="Normalized complexity",
    showgrid=True
)


fig.show(
    config={
        "scrollZoom": True,
        "displaylogo": False
    }
)


# ------------------------------------------------------------
# Language legend
# ------------------------------------------------------------

print()
print("LANGUAGE INDEX")
print("-" * 70)

display(
    plot_data[
        [
            "language_index",
            "language",
            "language_name"
        ]
    ]
    .sort_values(
        "language_index"
    )
)


LANGUAGE INDEX
----------------------------------------------------------------------


,language_index,language,language_name
180,0,abk,Abkhaz
157,1,atj,Achehnese
26,2,acu,Achuar-Shiwiar
119,3,gax,Afaan Oromo (Oromiffa)
142,4,afk,Afrikaans
...,...,...,...
6,230,rmy1,Vlach
25,231,frn1,Walloon/Wallon
50,232,guc,Wayuu
89,234,ccx,Zhuang


In [29]:
# ============================================================
# INTERACTIVE RAW COMPLEXITY VS TEXT LENGTH
# ============================================================

# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------

# Choose one font using its font_id.
#
# To see available fonts:
# display(font_table)

selected_font_id = analysis_fonts[0]


# ------------------------------------------------------------
# Prepare plot data
# ------------------------------------------------------------

plot_data = (
    text_complexity[
        text_complexity[
            "font_id"
        ] == selected_font_id
    ]
    .merge(
        language_table[
            [
                "language_index",
                "language"
            ]
        ],
        on="language",
        how="left",
        validate="many_to_one"
    )
    .dropna(
        subset=[
            "language_index",
            "nonwhitespace_characters",
            "total_complexity"
        ]
    )
    .copy()
)


plot_data[
    "language_index"
] = (
    plot_data[
        "language_index"
    ]
    .astype(int)
)


# ------------------------------------------------------------
# Get font name
# ------------------------------------------------------------

selected_font_name = (
    font_names.loc[
        selected_font_id
    ]
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig = px.scatter(
    plot_data,
    x="nonwhitespace_characters",
    y="total_complexity",
    text="language_index",
    hover_name="language_name",
    hover_data={
        "language_index": True,
        "language": True,
        "nonwhitespace_characters": ":,",
        "total_complexity": ":.6f"
    },
    labels={
        "nonwhitespace_characters":
            "Text length (non-whitespace characters)",
        "total_complexity":
            "Raw total complexity",
        "language_index":
            "Language index",
        "language":
            "Language code"
    },
    title=(
        "Raw Text Complexity vs. Text Length"
        f"<br>{selected_font_name}"
    )
)


# ------------------------------------------------------------
# Number labels
# ------------------------------------------------------------

fig.update_traces(
    marker=dict(
        size=7,
        opacity=0.75
    ),
    textposition="top center",
    textfont=dict(
        size=9
    )
)


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

fig.update_layout(
    width=1400,
    height=750,
    showlegend=False
)


fig.update_xaxes(
    title=(
        "Text length "
        "(non-whitespace characters, log scale)"
    ),
    type="log",
    showgrid=True
)


fig.update_yaxes(
    title="Raw total complexity",
    showgrid=True
)


fig.show(
    config={
        "scrollZoom": True,
        "displaylogo": False
    }
)


# ------------------------------------------------------------
# Language legend
# ------------------------------------------------------------

print()
print("LANGUAGE INDEX")
print("-" * 70)

display(
    plot_data[
        [
            "language_index",
            "language",
            "language_name"
        ]
    ]
    .sort_values(
        "language_index"
    )
)


LANGUAGE INDEX
----------------------------------------------------------------------


,language_index,language,language_name
0,0,abk,Abkhaz
10,1,atj,Achehnese
1,2,acu,Achuar-Shiwiar
61,3,gax,Afaan Oromo (Oromiffa)
2,4,afk,Afrikaans
...,...,...,...
151,230,rmy1,Vlach
59,231,frn1,Walloon/Wallon
68,232,guc,Wayuu
31,234,ccx,Zhuang


In [30]:
# ============================================================
# INTERACTIVE NORMALIZED COMPLEXITY VS TEXT LENGTH
# ============================================================

# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------

# Choose one font using its font_id.
#
# To see available fonts:
# display(font_table)

selected_font_id = analysis_fonts[0]


# ------------------------------------------------------------
# Prepare plot data
# ------------------------------------------------------------

plot_data = (
    text_complexity[
        text_complexity[
            "font_id"
        ] == selected_font_id
    ]
    .merge(
        language_table[
            [
                "language_index",
                "language"
            ]
        ],
        on="language",
        how="left",
        validate="many_to_one"
    )
    .dropna(
        subset=[
            "language_index",
            "nonwhitespace_characters",
            "normalized_complexity"
        ]
    )
    .copy()
)


plot_data[
    "language_index"
] = (
    plot_data[
        "language_index"
    ]
    .astype(int)
)


# ------------------------------------------------------------
# Get font name
# ------------------------------------------------------------

selected_font_name = (
    font_names.loc[
        selected_font_id
    ]
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig = px.scatter(
    plot_data,
    x="nonwhitespace_characters",
    y="normalized_complexity",
    text="language_index",
    hover_name="language_name",
    hover_data={
        "language_index": True,
        "language": True,
        "nonwhitespace_characters": ":,",
        "normalized_complexity": ":.6f"
    },
    labels={
        "nonwhitespace_characters":
            "Text length (non-whitespace characters)",
        "normalized_complexity":
            "Normalized complexity",
        "language_index":
            "Language index",
        "language":
            "Language code"
    },
    title=(
        "Normalized Text Complexity vs. Text Length"
        f"<br>{selected_font_name}"
    )
)


# ------------------------------------------------------------
# Number labels
# ------------------------------------------------------------

fig.update_traces(
    marker=dict(
        size=7,
        opacity=0.75
    ),
    textposition="top center",
    textfont=dict(
        size=9
    )
)


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

fig.update_layout(
    width=1400,
    height=750,
    showlegend=False
)


fig.update_xaxes(
    title=(
        "Text length "
        "(non-whitespace characters, log scale)"
    ),
    type="log",
    showgrid=True
)


fig.update_yaxes(
    title="Normalized complexity",
    showgrid=True
)


fig.show(
    config={
        "scrollZoom": True,
        "displaylogo": False
    }
)


# ------------------------------------------------------------
# Language legend
# ------------------------------------------------------------

print()
print("LANGUAGE INDEX")
print("-" * 70)

display(
    plot_data[
        [
            "language_index",
            "language",
            "language_name"
        ]
    ]
    .sort_values(
        "language_index"
    )
)


LANGUAGE INDEX
----------------------------------------------------------------------


,language_index,language,language_name
0,0,abk,Abkhaz
10,1,atj,Achehnese
1,2,acu,Achuar-Shiwiar
61,3,gax,Afaan Oromo (Oromiffa)
2,4,afk,Afrikaans
...,...,...,...
151,230,rmy1,Vlach
59,231,frn1,Walloon/Wallon
68,232,guc,Wayuu
31,234,ccx,Zhuang


In [31]:
# ============================================================
# ENGLISH COMPLEXITY — FONTS VS INTERNATIONAL MORSE CODE
# ============================================================

# This comparison uses only the English source-character
# occurrences that are representable in International Morse
# Code.
#
# Unsupported English characters were excluded from BOTH:
#     1. the Morse calculation
#     2. the matched English-font calculations
#
# Therefore, every bar represents the same source-character
# population encoded using a different visual representation.


# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------

comparison_system = (
    "International Morse Code"
)


# ------------------------------------------------------------
# Prepare plot data
# ------------------------------------------------------------

plot_data = (
    english_representation_comparison[
        english_representation_comparison[
            "comparison_system"
        ] == comparison_system
    ]
    .dropna(
        subset=[
            "normalized_complexity"
        ]
    )
    .copy()
)


# ------------------------------------------------------------
# Sort from least → most complex
# ------------------------------------------------------------

plot_data = (
    plot_data
    .sort_values(
        "normalized_complexity",
        ascending=True
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig = px.bar(
    plot_data,
    x="representation_name",
    y="normalized_complexity",
    color="representation_type",
    text="normalized_complexity",
    hover_data={
        "representation_name": True,
        "representation_type": True,
        "normalized_complexity": ":.6f",
        "matched_characters": ":,",
        "matched_unique_characters": ":,"
    },
    labels={
        "representation_name":
            "Visual representation",
        "normalized_complexity":
            "Normalized complexity",
        "representation_type":
            "Representation type",
        "matched_characters":
            "Matched character occurrences",
        "matched_unique_characters":
            "Matched unique characters"
    },
    title=(
        "English Visual Complexity"
        "<br>Fonts vs. International Morse Code"
    )
)


# ------------------------------------------------------------
# Show complexity values
# ------------------------------------------------------------

fig.update_traces(
    texttemplate="%{y:.4f}",
    textposition="outside"
)


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

fig.update_layout(
    width=1100,
    height=700,

    legend=dict(
        title="Representation type",
        x=1.02,
        y=1
    ),

    margin=dict(
        r=220
    )
)


fig.update_xaxes(
    title="Visual representation"
)


fig.update_yaxes(
    title="Normalized complexity",
    rangemode="tozero",
    showgrid=True
)


fig.show(
    config={
        "displaylogo": False
    }
)

In [32]:
# ============================================================
# ENGLISH COMPLEXITY — FONTS VS UNIFIED ENGLISH BRAILLE
# ============================================================

# This comparison uses only the English source-character
# occurrences that are representable in Unified English
# Braille.
#
# Unsupported English characters were excluded from BOTH:
#     1. the Braille calculation
#     2. the matched English-font calculations
#
# Therefore, every bar represents the same source-character
# population encoded using a different visual representation.


# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------

comparison_system = (
    "Unified English Braille"
)


# ------------------------------------------------------------
# Prepare plot data
# ------------------------------------------------------------

plot_data = (
    english_representation_comparison[
        english_representation_comparison[
            "comparison_system"
        ] == comparison_system
    ]
    .dropna(
        subset=[
            "normalized_complexity"
        ]
    )
    .copy()
)


# ------------------------------------------------------------
# Sort from least → most complex
# ------------------------------------------------------------

plot_data = (
    plot_data
    .sort_values(
        "normalized_complexity",
        ascending=True
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig = px.bar(
    plot_data,
    x="representation_name",
    y="normalized_complexity",
    color="representation_type",
    text="normalized_complexity",
    hover_data={
        "representation_name": True,
        "representation_type": True,
        "normalized_complexity": ":.6f",
        "matched_characters": ":,",
        "matched_unique_characters": ":,"
    },
    labels={
        "representation_name":
            "Visual representation",
        "normalized_complexity":
            "Normalized complexity",
        "representation_type":
            "Representation type",
        "matched_characters":
            "Matched character occurrences",
        "matched_unique_characters":
            "Matched unique characters"
    },
    title=(
        "English Visual Complexity"
        "<br>Fonts vs. Unified English Braille"
    )
)


# ------------------------------------------------------------
# Show complexity values
# ------------------------------------------------------------

fig.update_traces(
    texttemplate="%{y:.4f}",
    textposition="outside"
)


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

fig.update_layout(
    width=1100,
    height=700,

    legend=dict(
        title="Representation type",
        x=1.02,
        y=1
    ),

    margin=dict(
        r=220
    )
)


fig.update_xaxes(
    title="Visual representation"
)


fig.update_yaxes(
    title="Normalized complexity",
    rangemode="tozero",
    showgrid=True
)


fig.show(
    config={
        "displaylogo": False
    }
)

In [33]:
# ============================================================
# REPRESENTATION COMPLEXITY VS AVERAGE ENGLISH FONT COMPLEXITY
# ============================================================

# For each representation system:
#
#     1. Calculate the mean normalized complexity across the
#        matched English font representations.
#
#     2. Compare the encoded representation complexity
#        (Morse or Braille) with that English-font mean.
#
# Percent change:
#
#     ((representation - English mean) / English mean) * 100
#
# Positive values:
#     representation is more complex than average English font
#
# Negative values:
#     representation is less complex than average English font


representation_change_records = []


for comparison_system in [
    "International Morse Code",
    "Unified English Braille"
]:

    system_data = (
        english_representation_comparison[
            english_representation_comparison[
                "comparison_system"
            ] == comparison_system
        ]
        .copy()
    )


    # --------------------------------------------------------
    # English font observations
    # --------------------------------------------------------

    english_font_data = (
        system_data[
            system_data[
                "representation_type"
            ] == "Font"
        ]
        .dropna(
            subset=[
                "normalized_complexity"
            ]
        )
        .copy()
    )


    if english_font_data.empty:

        raise ValueError(
            f"No matched English font data found for "
            f"{comparison_system}."
        )


    english_mean_complexity = (
        english_font_data[
            "normalized_complexity"
        ]
        .mean()
    )


    # --------------------------------------------------------
    # Encoded representation
    # --------------------------------------------------------

    representation_data = (
        system_data[
            system_data[
                "representation_type"
            ] == "Encoded representation"
        ]
        .dropna(
            subset=[
                "normalized_complexity"
            ]
        )
        .copy()
    )


    if len(representation_data) != 1:

        raise ValueError(
            f"Expected exactly one encoded representation "
            f"row for {comparison_system}, but found "
            f"{len(representation_data)}."
        )


    representation_complexity = (
        representation_data[
            "normalized_complexity"
        ]
        .iloc[0]
    )


    representation_name = (
        representation_data[
            "representation_name"
        ]
        .iloc[0]
    )


    # --------------------------------------------------------
    # Percent change
    # --------------------------------------------------------

    percent_change = (
        (
            representation_complexity
            -
            english_mean_complexity
        )
        /
        english_mean_complexity
        *
        100
    )


    representation_change_records.append(
        {
            "comparison_system":
                comparison_system,

            "representation_name":
                representation_name,

            "english_fonts":
                len(
                    english_font_data
                ),

            "mean_english_font_complexity":
                english_mean_complexity,

            "representation_complexity":
                representation_complexity,

            "absolute_change":
                (
                    representation_complexity
                    -
                    english_mean_complexity
                ),

            "percent_change":
                percent_change
        }
    )


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

representation_complexity_change = (
    pd.DataFrame(
        representation_change_records
    )
)


print()
print("=" * 70)
print(
    "REPRESENTATION COMPLEXITY VS "
    "AVERAGE ENGLISH FONT COMPLEXITY"
)
print("=" * 70)
print()


display(
    representation_complexity_change
)


# ------------------------------------------------------------
# Human-readable summary
# ------------------------------------------------------------

for _, row in (
    representation_complexity_change
    .iterrows()
):

    direction = (
        "higher"
        if row[
            "percent_change"
        ] > 0
        else "lower"
    )


    print(
        f"{row['representation_name']}: "
        f"{abs(row['percent_change']):.2f}% "
        f"{direction} complexity than the mean "
        f"of the matched English fonts."
    )


REPRESENTATION COMPLEXITY VS AVERAGE ENGLISH FONT COMPLEXITY



,comparison_system,representation_name,english_fonts,mean_english_font_complexity,representation_complexity,absolute_change,percent_change
0,International Morse Code,International Morse Code,5,0.187517,0.151779,-0.035737,-19.058232
1,Unified English Braille,Unified English Braille,5,0.187514,0.191222,0.003708,1.977317


International Morse Code: 19.06% lower complexity than the mean of the matched English fonts.
Unified English Braille: 1.98% higher complexity than the mean of the matched English fonts.
